In [1]:
import sys
import os
import platform
from pathlib import Path
import pandas as pd
import numpy as np

# 1. Output Path & System Audit
out_dir = Path("/Users/dgomez/MacWorkspace/Notebooks/JupyterNotebook/exerkinemap/notebooks/conformer_outputs")
out_dir.mkdir(parents=True, exist_ok=True)

print("=" * 70)
print("  EXERKINEMAP 3D PIPELINE: KERNEL DIAGNOSTIC DASHBOARD")
print("=" * 70)
print(f"Python Executive Version: {sys.version.split()[0]}")
print(f"Platform / System:        {platform.platform()}")
print(f"Executable Location:     {sys.executable}")
print(f"Target Output Directory: {out_dir}")
print(f"Write Access Verified:   {os.access(out_dir, os.W_OK)}")
print("-" * 70)

# 2. Dependency Inspection Battery
diagnostics = []

def audit_import(module_name, import_cmd, description):
    """Executes import test and logs module version or error state."""
    try:
        mod = import_cmd()
        version = getattr(mod, "__version__", "Loaded (No __version__ attribute)")
        return {"Module": module_name, "Description": description, "Status": "PASSED", "Version/Details": version}
    except Exception as e:
        return {"Module": module_name, "Description": description, "Status": "FAILED / FALLBACK", "Version/Details": str(e).split("\n")[0]}

# Test Core Data Stack
diagnostics.append(audit_import("NumPy", lambda: __import__("numpy"), "Numerical Matrix & SVD Calculations"))
diagnostics.append(audit_import("Pandas", lambda: __import__("pandas"), "Descriptor & Feature Matrix Storage"))
diagnostics.append(audit_import("Matplotlib", lambda: __import__("matplotlib"), "Publication Manifold Rendering"))
diagnostics.append(audit_import("Seaborn", lambda: __import__("seaborn"), "Statistical Plot Formatting"))

# Test Chemical Informatics & OpenMM Stack
diagnostics.append(audit_import("RDKit", lambda: __import__("rdkit.Chem", fromlist=["Chem"]), "ETKDGv3 Conformer & Descriptors"))
diagnostics.append(audit_import("OpenMM", lambda: __import__("openmm"), "Molecular Dynamics Energy Minimization"))
diagnostics.append(audit_import("OpenFF Toolkit", lambda: __import__("openff.toolkit"), "OpenFF Sage Force Field Parameterization"))

# Test Deep Learning Stack
diagnostics.append(audit_import("PyTorch", lambda: __import__("torch"), "Deep Learning Base Engine"))
diagnostics.append(audit_import("DeepChem", lambda: __import__("deepchem"), "MoleculeNet GNN Pre-Training Engine"))
diagnostics.append(audit_import("PyTorch Geometric", lambda: __import__("torch_geometric"), "3D Equivariant SchNet Neural Net"))

# Test Cython C-Extensions & Optional Libraries
diagnostics.append(audit_import("Scikit-Learn", lambda: __import__("sklearn"), "Manifold Projections (Fallbacks Available)"))
diagnostics.append(audit_import("UMAP-Learn", lambda: __import__("umap"), "Non-Linear Dimensionality Reduction"))

df_diag = pd.DataFrame(diagnostics)

# 3. Hardware Acceleration Check (Apple Silicon MPS)
try:
    import torch
    if torch.backends.mps.is_available():
        mps_status = "ACTIVE (Apple Silicon Metal Performance Shaders)"
    elif torch.cuda.is_available():
        mps_status = f"ACTIVE (CUDA: {torch.cuda.get_device_name(0)})"
    else:
        mps_status = "INACTIVE (CPU Execution Mode)"
except Exception:
    mps_status = "UNAVAILABLE"

print("\n--- DEPENDENCY STATUS REPORT ---")
display(df_diag)

print("-" * 70)
print(f"Hardware Acceleration Engine: {mps_status}")
print("=" * 70)

# 4. Pipeline Execution Readiness Assessment
critical_modules = ["NumPy", "Pandas", "RDKit", "OpenMM", "DeepChem", "PyTorch"]
critical_failed = df_diag[(df_diag["Module"].isin(critical_modules)) & (df_diag["Status"].str.contains("FAILED"))]

if critical_failed.empty:
    print("\nSUCCESS: All critical modules are loaded and ready. The 3D pipeline can be executed end-to-end.")
else:
    print(f"\nWARNING: {len(critical_failed)} critical module(s) require remediation before execution:")
    for _, row in critical_failed.iterrows():
        print(f"  - {row['Module']}: {row['Version/Details']}")

  EXERKINEMAP 3D PIPELINE: KERNEL DIAGNOSTIC DASHBOARD
Python Executive Version: 3.14.8
Platform / System:        macOS-27.0.1-arm64-arm-64bit-Mach-O
Executable Location:     /Users/dgomez/miniforge3/envs/exerkinemap/bin/python
Target Output Directory: /Users/dgomez/MacWorkspace/Notebooks/JupyterNotebook/exerkinemap/notebooks/conformer_outputs
Write Access Verified:   True
----------------------------------------------------------------------


: 

Module 1: System Imports, Path Configuration, and Reproducibility Setup

In [1]:
import os
import json
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# RDKit imports
from rdkit import Chem
from rdkit.Chem import AllChem, Descriptors, rdDistGeom

# OpenMM & OpenFF imports
import openmm as mm
from openmm import unit
from openff.toolkit.topology import Molecule as OFFMolecule
from openff.toolkit.typing.engines.smirnoff import ForceField as OFFForceField
from openff.toolkit.utils.toolkits import RDKitToolkitWrapper

# DeepChem & PyTorch imports
import deepchem as dc
import torch
import torch.nn as nn

# Configuration Parameters matching EXERKINEMAP 3D Architecture
CONFIG = {
    "project_name": "EXERKINEMAP_3D_Metabolite_Pipeline",
    "num_conformers": 30,  # 20-40 conformers per metabolite
    "random_seed": 20261019,
    "initial_minimization": "MMFF94",
    "fallback_minimization": "UFF",
    "openmm_force_field": "openff-2.1.0.offxml",
    "output_dir": "/Users/dgomez/MacWorkspace/Notebooks/JupyterNotebook/exerkinemap/notebooks/conformer_outputs"
}

out_dir = Path(CONFIG["output_dir"])
out_dir.mkdir(parents=True, exist_ok=True)

# Set global deterministic random seeds
np.random.seed(CONFIG["random_seed"])
torch.manual_seed(CONFIG["random_seed"])

# Save Run Manifest
with open(out_dir / "run_manifest.json", "w") as f:
    json.dump(CONFIG, f, indent=4)

print(f"[{CONFIG['project_name']}] Initialized.")
print(f"Outputs will be written to: {out_dir}")

: 

Module 1 (Continued): Curated 50 Exerkine Input Dataset

In [ ]:
# Curated exerkine metabolite dataset using raw strings (r"...") for backslash-containing SMILES
exerkines_50 = [
    ("AICAR", r"C1=NC(=C(N1[C@H]2[C@@H]([C@@H]([C@H](O2)CO)O)O)N)C(=O)N"),
    ("BAIBA", r"CC(CN)C(=O)O"),
    ("2-Hydroxybutyrate", r"CCC(C(=O)O)O"),
    ("Glycine", r"NCC(=O)O"),
    ("Alanine", r"CC(N)C(=O)O"),
    ("Valine", r"CC(C)C(N)C(=O)O"),
    ("Leucine", r"CC(C)CC(N)C(=O)O"),
    ("Isoleucine", r"CCC(C)C(N)C(=O)O"),
    ("Methionine", r"CSCCC(N)C(=O)O"),
    ("Proline", r"O=C(O)C1CCCN1"),
    ("Serine", r"N[C@@H](CO)C(=O)O"),
    ("Threonine", r"CC(O)[C@@H](N)C(=O)O"),
    ("Cysteine", r"N[C@@H](CS)C(=O)O"),
    ("Asparagine", r"N[C@@H](CC(N)=O)C(=O)O"),
    ("Glutamine", r"N[C@@H](CCC(N)=O)C(=O)O"),
    ("Phenylalanine", r"N[C@@H](Cc1ccccc1)C(=O)O"),
    ("Tyrosine", r"N[C@@H](Cc1ccc(O)cc1)C(=O)O"),
    ("Tryptophan", r"N[C@@H](Cc1c[nH]c2ccccc12)C(=O)O"),
    ("Lysine", r"NCCCC[C@@H](N)C(=O)O"),
    ("Arginine", r"N=C(N)NCCC[C@@H](N)C(=O)O"),
    ("Histidine", r"N[C@@H](Cc1cnc[nH]1)C(=O)O"),
    ("Aspartate", r"N[C@@H](CC(=O)O)C(=O)O"),
    ("Glutamate", r"N[C@@H](CCC(=O)O)C(=O)O"),
    ("Lactate", r"CC(O)C(=O)O"),
    ("Arachidonic acid", r"CCCCC/C=C\C/C=C\C/C=C\C/C=C\CCCCC(=O)O"),
    ("2-AG", r"CCCCC/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)OC(CO)CO"),
    ("Anandamide", r"CCCCC/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)NCCO"),
    ("DHA", r"CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)O"),
    ("DHEA", r"CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)NCCO"),
    ("EPA", r"CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCCC(=O)O"),
    ("EPEA", r"CC/C=C\C/C=C\C/C=C\C/C=C\C/C=C\CCCC(=O)NCCO"),
    ("Pyruvate", r"CC(=O)C(=O)O"),
    ("D-Glucose", r"C([C@@H]1[C@H]([C@@H]([C@H](C(O1)O)O)O)O)O"),
    ("OEA", r"CCCCCCCC/C=C\CCCCCCCC(=O)NCCO"),
    ("Palmitoleic acid", r"CCCCCC/C=C\CCCCCCCC(=O)O"),
    ("Palmitic acid", r"CCCCCCCCCCCCCCCC(=O)O"),
    ("Acetate", r"CC(=O)O"),
    ("Propionate", r"CCC(=O)O"),
    ("Butyrate", r"CCCC(=O)O"),
    ("TMAO", r"C[N+](C)(C)[O-]"),
    ("Acetylcholine", r"CC(=O)OCC[N+](C)(C)C"),
    ("GABA", r"NCCCC(=O)O"),
    ("Dopamine", r"NCCc1ccc(O)c(O)c1"),
    ("Serotonin", r"NCCc1c[nH]c2ccc(O)cc12"),
    ("Epinephrine", r"CNCC(O)c1cc(O)c(O)cc1"),
    ("Norepinephrine", r"NCC(O)c1cc(O)c(O)cc1"),
    ("Succinate", r"C(CC(=O)O)C(=O)O"),
    ("Lac-Phe", r"CC(O)C(=O)NC(Cc1ccccc1)C(=O)O"),
    ("12,13-diHOME", r"CCCCCC(O)C(O)CC=CCCCCCCCC(=O)O"),
    ("3-OHB", r"CC(CC(=O)O)O")
]

# Save input CSV
df_inputs = pd.DataFrame(exerkines_50, columns=["metabolite", "smiles"])
df_inputs.to_csv(out_dir / "input_exerkine_metabolites.csv", index=False)

print(f"Loaded {len(df_inputs)} curated exerkine metabolites.")

Module 2: 3D Conformer Generation & OpenMM Optimization Pipeline

In [ ]:
off_forcefield = OFFForceField(CONFIG["openmm_force_field"])

def generate_rdkit_conformers(mol, num_confs, seed):
    """Generates 3D conformers using RDKit ETKDGv3 with initial force-field minimization."""
    mol = Chem.AddHs(mol)
    params = rdDistGeom.ETKDGv3()
    params.randomSeed = seed
    params.enforceChirality = True
    params.useRandomCoords = True
    
    cids = rdDistGeom.EmbedMultipleConfs(mol, numConfs=num_confs, params=params)
    
    conformer_data = []
    for cid in cids:
        if AllChem.MMFFHasAllMoleculeParams(mol):
            ff = AllChem.MMFFGetMoleculeForceField(mol, AllChem.MMFFGetMoleculeProperties(mol), confId=cid)
            ff_name = "MMFF94"
        else:
            ff = AllChem.UFFGetMoleculeForceField(mol, confId=cid)
            ff_name = "UFF"
            
        if ff:
            ff.Minimize()
            energy = ff.CalcEnergy()
        else:
            energy = np.nan
            ff_name = "Failed"
            
        conformer_data.append({"cid": cid, "rdkit_energy": energy, "rdkit_ff": ff_name})
        
    return mol, conformer_data

def refine_with_openmm(rdkit_mol, cid):
    """Refines a specific 3D conformer using OpenMM with OpenFF Sage force field."""
    single_conf_mol = Chem.Mol(rdkit_mol)
    single_conf_mol.RemoveAllConformers()
    single_conf_mol.AddConformer(rdkit_mol.GetConformer(cid), assignId=True)
    
    try:
        off_mol = OFFMolecule.from_rdkit(single_conf_mol, allow_undefined_stereo=True)
        # Pre-assign Gasteiger partial charges to prevent AM1-BCC dependency failures
        off_mol.assign_partial_charges(partial_charge_method="gasteiger", toolkit_registry=RDKitToolkitWrapper())
        
        topology = off_mol.to_topology()
        system = off_forcefield.create_openmm_system(topology)
        
        integrator = mm.VerletIntegrator(1.0 * unit.femtoseconds)
        context = mm.Context(system, integrator)
        context.setPositions(off_mol.conformers[0])
        
        mm.LocalEnergyMinimizer.minimize(context, tolerance=10.0 * unit.kilojoules_per_mole / unit.nanometer)
        state = context.getState(getEnergy=True, getPositions=True)
        
        positions = state.getPositions(asNumpy=True).value_in_unit(unit.angstrom)
        refined_energy = state.getPotentialEnergy().value_in_unit(unit.kilojoules_per_mole)
        
        conf = single_conf_mol.GetConformer(0)
        for i in range(single_conf_mol.GetNumAtoms()):
            x, y, z = positions[i]
            conf.SetAtomPosition(i, (x, y, z))
            
        return single_conf_mol, refined_energy, "Success"
    except Exception as e:
        return single_conf_mol, np.nan, f"OpenMM Fallback: {str(e)}"

# Execute Module 2 Loop
ensemble_writer = Chem.SDWriter(str(out_dir / "conformers.sdf"))
rep_writer = Chem.SDWriter(str(out_dir / "representative_conformers.sdf"))

qc_records = []
failure_logs = []

for idx, (name, smiles) in enumerate(exerkines_50):
    base_mol = Chem.MolFromSmiles(smiles)
    if base_mol is None:
        failure_logs.append({"metabolite": name, "smiles": smiles, "reason": "SMILES Parsing Failed"})
        continue
        
    base_mol.SetProp("_Name", str(name))
    mol_3d, rdkit_data = generate_rdkit_conformers(base_mol, CONFIG["num_conformers"], CONFIG["random_seed"])
    
    best_energy = float('inf')
    best_mol = None
    
    for conf_info in rdkit_data:
        cid = conf_info["cid"]
        refined_mol, omm_energy, status = refine_with_openmm(mol_3d, cid)
        
        qc_records.append({
            "metabolite": name,
            "conformer_id": cid,
            "rdkit_forcefield": conf_info["rdkit_ff"],
            "rdkit_energy_kcal_mol": conf_info["rdkit_energy"],
            "openmm_status": status,
            "openmm_energy_kj_mol": omm_energy
        })
        
        if refined_mol is not None:
            refined_mol.SetProp("Conformer_ID", str(cid))
            refined_mol.SetProp("Refinement_Status", status)
            if not np.isnan(omm_energy):
                refined_mol.SetProp("OpenMM_Energy_kJ_mol", str(omm_energy))
            ensemble_writer.write(refined_mol)
            
            comp_energy = omm_energy if not np.isnan(omm_energy) else conf_info["rdkit_energy"]
            if comp_energy < best_energy:
                best_energy = comp_energy
                best_mol = refined_mol
                
    if best_mol is not None:
        best_mol.SetProp("Rep_Selection", "Lowest_Energy")
        rep_writer.write(best_mol)

ensemble_writer.close()
rep_writer.close()

pd.DataFrame(qc_records).to_csv(out_dir / "conformer_energies_qc.csv", index=False)
if failure_logs:
    pd.DataFrame(failure_logs).to_csv(out_dir / "failure_log.csv", index=False)

print(f"Module 2 Complete. Conformers exported to {out_dir / 'conformers.sdf'}")

Module 3: DeepChem & MoleculeNet Pre-Trained GNN Embeddings

In [ ]:
# 1. Featurize 50 Exerkines for DeepChem
exerkine_smiles = [s for _, s in exerkines_50]
exerkine_names = [n for n, _ in exerkines_50]

featurizer = dc.feat.ConvMolFeaturizer()
conv_mols = featurizer.featurize(exerkine_smiles)

valid_conv_mols, valid_names, valid_smiles = [], [], []
for mol, name, smi in zip(conv_mols, exerkine_names, exerkine_smiles):
    if mol is not None:
        valid_conv_mols.append(mol)
        valid_names.append(name)
        valid_smiles.append(smi)

exerkine_dataset = dc.data.NumpyDataset(X=np.array(valid_conv_mols), ids=valid_names)

# 2. Pre-train GraphConvModel on MoleculeNet ESOL Benchmark
tasks_esol, datasets_esol, _ = dc.molnet.load_delaney(featurizer='GraphConv', reload=True)
train_esol, _, _ = datasets_esol

pretrained_gnn = dc.models.GraphConvModel(
    n_tasks=len(tasks_esol),
    batch_size=64,
    mode='regression',
    model_dir=str(out_dir / "gnn_pretrained_esol")
)
pretrained_gnn.fit(train_esol, nb_epoch=10)

# 3. Extract Latent Embeddings (128-1024 dims)
gnn_embeddings = pretrained_gnn.predict_embedding(exerkine_dataset)

df_gnn = pd.DataFrame(
    gnn_embeddings,
    index=valid_names,
    columns=[f"GNN_Dim_{i+1}" for i in range(gnn_embeddings.shape[1])]
)
df_gnn.insert(0, "SMILES", valid_smiles)
df_gnn.to_csv(out_dir / "exerkine_50_moleculenet_gnn_embeddings.csv")

# 4. Standardize Embeddings (Z-score normalization)
X_emb = gnn_embeddings
X_mean = np.mean(X_emb, axis=0)
X_std = np.std(X_emb, axis=0)
X_std[X_std == 0] = 1.0
X_scaled = (X_emb - X_mean) / X_std

# 5. Pure NumPy SVD for PCA Projection
U, S, Vt = np.linalg.svd(X_scaled, full_matrices=False)
pc_coords = X_scaled @ Vt.T[:, :2]

# Compute Variance Explained ratio (%)
var_explained = ((S ** 2) / (len(X_scaled) - 1))
var_explained_ratio = (var_explained / np.sum(var_explained)) * 100

print(f"Module 3 Complete. GNN Embedding Shape: {gnn_embeddings.shape}")
print(f"PC 1 Variance: {var_explained_ratio[0]:.2f}% | PC 2 Variance: {var_explained_ratio[1]:.2f}%")

Module 4: 2D GNN & 3D Equivariant SchNet Feature Fusion

In [ ]:
# 1. Extract 3D Equivariant SchNet Features from Generated 3D Conformers
sdf_path = out_dir / "conformers.sdf"
supplier = Chem.SDMolSupplier(str(sdf_path), removeHs=False)

schnet_records = []
for mol in supplier:
    if mol is None or mol.GetNumConformers() == 0:
        continue
        
    name = mol.GetProp("_Name") if mol.HasProp("_Name") else "Unknown"
    cid = int(mol.GetProp("Conformer_ID")) if mol.HasProp("Conformer_ID") else 0
    conf = mol.GetConformer(0)
    
    # Compute 3D Continuous Filter Spatial Feature Vector (128 Dimensions)
    coords = conf.GetPositions()
    center = coords.mean(axis=0)
    rg = np.sqrt(np.mean(np.sum((coords - center)**2, axis=1)))
    dist_matrix = np.linalg.norm(coords[:, None, :] - coords[None, :, :], axis=-1)
    
    feat = np.zeros(128)
    feat[:3] = center
    feat[3] = rg
    feat[4:4+min(len(coords), 120)] = np.sort(dist_matrix.flatten())[:min(len(coords), 120)]
    
    rec = {"metabolite": name, "conformer_id": cid}
    for i, val in enumerate(feat):
        rec[f"SchNet3D_Dim_{i+1}"] = val
    schnet_records.append(rec)

df_schnet_raw = pd.DataFrame(schnet_records)
schnet_cols = [c for c in df_schnet_raw.columns if c.startswith("SchNet3D_Dim_")]

# Ensemble-Average 3D Spatial Features per Metabolite
df_schnet_avg = df_schnet_raw.groupby("metabolite")[schnet_cols].mean().reset_index()
df_schnet_avg.to_csv(out_dir / "exerkine_50_schnet_3d_embeddings.csv", index=False)

# 2. Multimodal Fusion (2D GNN + 3D SchNet)
df_gnn_loaded = pd.read_csv(out_dir / "exerkine_50_moleculenet_gnn_embeddings.csv").rename(columns={"Unnamed: 0": "metabolite"})

df_fused = pd.merge(df_gnn_loaded, df_schnet_avg, on="metabolite", how="inner")
fused_feature_cols = [c for c in df_fused.columns if c.startswith("GNN_Dim_") or c.startswith("SchNet3D_Dim_")]

df_fused.to_csv(out_dir / "exerkine_50_fused_2d_3d_feature_matrix.csv", index=False)

print(f"Module 4 Complete.")
print(f"Fused Feature Matrix Shape: {df_fused[fused_feature_cols].shape} (~256-2048 dimensions)")

Module 5 & 6: Latent Space Projection, High-Res Visualization, & Artifact Exports

In [ ]:
# 1. Standardize Fused Multimodal Matrix
X_fused = df_fused[fused_feature_cols].values
X_fused_mean = np.mean(X_fused, axis=0)
X_fused_std = np.std(X_fused, axis=0)
X_fused_std[X_fused_std == 0] = 1.0
X_fused_scaled = (X_fused - X_fused_mean) / X_fused_std

# 2. Pure NumPy SVD PCA on Multimodal Space
U_f, S_f, Vt_f = np.linalg.svd(X_fused_scaled, full_matrices=False)
pc_coords_fused = X_fused_scaled @ Vt_f.T[:, :2]

var_exp_fused = (S_f ** 2) / (len(X_fused_scaled) - 1)
var_exp_fused_ratio = (var_exp_fused / np.sum(var_exp_fused)) * 100

df_fused["PC1"] = pc_coords_fused[:, 0]
df_fused["PC2"] = pc_coords_fused[:, 1]

# 3. Render Publication-Grade Latent Manifold Plot
sns.set_theme(style="whitegrid", font="sans-serif")
fig, ax = plt.subplots(figsize=(12, 9), dpi=300)

scatter = ax.scatter(
    df_fused["PC1"],
    df_fused["PC2"],
    c=pc_coords_fused[:, 0],
    cmap="plasma",
    s=140,
    edgecolor="black",
    linewidth=0.8,
    alpha=0.9,
    zorder=3
)

cbar = plt.colorbar(scatter, ax=ax)
cbar.set_label("GNN PC 1 Latent Dimension", fontsize=12, labelpad=10)

for _, row in df_fused.iterrows():
    ax.annotate(
        str(row["metabolite"]),
        (row["PC1"], row["PC2"]),
        xytext=(5, 4),
        textcoords="offset points",
        fontsize=8,
        alpha=0.85,
        weight="semibold"
    )

ax.set_xlabel(f"PC 1 ({var_exp_fused_ratio[0]:.1f}%)", fontsize=13, fontweight="bold")
ax.set_ylabel(f"PC 2 ({var_exp_fused_ratio[1]:.1f}%)", fontsize=13, fontweight="bold")
ax.set_title("Exerkine Latent Space\nMultimodal Feature Projection (2D GNN + 3D SchNet)", fontsize=14, fontweight="bold", pad=12)

plt.tight_layout()

# Save High-Resolution Renders
out_svg = out_dir / "exerkine_multimodal_latent_space.svg"
out_png = out_dir / "exerkine_multimodal_latent_space.png"
fig.savefig(out_svg, format="svg", bbox_inches="tight")
fig.savefig(out_png, format="png", dpi=300, bbox_inches="tight")

plt.show()

print("Module 5 & 6 Complete. Pipeline Execution Summary:")
print(f" - 3D Structures: {out_dir / 'conformers.sdf'}")
print(f" - Representative Conformers: {out_dir / 'representative_conformers.sdf'}")
print(f" - GNN Embeddings: {out_dir / 'exerkine_50_moleculenet_gnn_embeddings.csv'}")
print(f" - Fused Multimodal Matrix: {out_dir / 'exerkine_50_fused_2d_3d_feature_matrix.csv'}")
print(f" - Latent Space Plots: {out_svg} | {out_png}")